Пример на датасете 1 000 000 строк: сравним одинаковую операцию в pandas и Polars - groupby + mean.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/synthetic_faers_1m_v3.csv')
df.head(10)

,case_id,receive_date,country,age,age_group,sex,weight_kg,suspect_drug,indication,route,...,concomitant_medications,medical_history,reporter_type,report_type,causality_assessment,alt_u_l,ast_u_l,bilirubin_mg_dl,creatinine_mg_dl,bun_mg_dl
0,E787C9A1FC17D940,2023-04-19,CHN,52,adult,Unknown,NaN,Hematocare,Hypothyroidism,Subcutaneous,...,Aspirin,NaN,Physician,Spontaneous,Probable/Likely,12.0,12.0,0.4,1.0,16.0
1,A29236736650CAB3,2021-10-30,POL,45,adult,Male,110.5,Osteostrong,Bipolar disorder,Intravenous,...,Hydrochlorothiazide,NaN,Consumer,Clinical trial,Probable/Likely,15.0,NaN,0.5,7.1,46.0
2,837A1C25D1B0ED72,2021-09-12,ITA,92,elderly,Female,NaN,Hematocare,COPD,Subcutaneous,...,NaN,Hyperlipidemia; Alcohol use disorder; Chronic ...,Physician,Spontaneous,Possible,340.0,NaN,9.5,0.9,15.0
3,BF1523DD99667C0D,2021-10-03,USA,47,adult,Male,115.0,Vasodilate,Schizophrenia,Ophthalmic,...,NaN,Peripheral vascular disease; Hypothyroidism; P...,Consumer,Spontaneous,Possible,45.0,NaN,1.1,1.2,9.0
4,F470F722F96A2FF7,2021-03-08,GBR,19,adult,Female,94.4,Cardiomax,Rheumatoid arthritis,Intramuscular,...,Pantoprazole; Hydrochlorothiazide,NaN,Pharmacist,Spontaneous,Unlikely,33.0,35.0,0.1,1.1,12.0
5,785B2438894CDCFA,2025-08-05,ESP,89,elderly,Male,111.7,Eyeprotect,Rheumatoid arthritis,Topical,...,Warfarin; Aspirin,Osteoarthritis; Depression,Other health professional,Literature,Possible,27.0,26.0,1.2,NaN,NaN
6,5A0796C50364BAA5,2023-04-29,CAN,58,adult,Unknown,58.1,Immunoshield,Type 2 diabetes,Topical,...,Atorvastatin; Ciprofloxacin,Chronic pain; Hypothyroidism; Heart failure,Lawyer,Spontaneous,Possible,28.0,16.0,1.1,1.2,11.0
7,BEA3CC1D4F66B39B,2022-04-26,NLD,47,adult,Unknown,NaN,Eyeprotect,Anxiety,Intravenous,...,Lisinopril,Hepatitis B,Consumer,Spontaneous,Possible,34.0,NaN,0.9,1.2,19.0
8,A3B710F35B3DBE9E,2022-04-13,BEL,78,elderly,Unknown,NaN,Thyroregul,Schizophrenia,Inhalation,...,Warfarin,Gastroesophageal reflux disease; Occupational ...,Lawyer,Spontaneous,Unlikely,27.0,32.0,0.4,NaN,10.0
9,73BDA39A3B797270,2021-10-29,AUT,9,child,Unknown,46.2,Cardiomax,Infection,Ophthalmic,...,Acetaminophen; Warfarin,Gastroesophageal reflux disease,Lawyer,Spontaneous,Possible,34.0,31.0,0.9,1.2,NaN


In [4]:
import time
import polars as pl

# pandas DataFrame у тебя уже есть: df
pdf = df

# создаем Polars DataFrame
pldf = pl.from_pandas(pdf)

In [5]:
def benchmark_many(func, name, repeats=5):
    times = []

    for _ in range(repeats):
        start = time.perf_counter()
        result = func()
        end = time.perf_counter()
        times.append(end - start)

    avg_time = sum(times) / len(times)

    print(f"{name}")
    print(f"Среднее время: {avg_time:.4f} секунд")
    print(f"Все замеры: {[round(t, 4) for t in times]}")
    print("-" * 50)

    return result

In [6]:
pandas_result_1 = benchmark_many(
    lambda: (
        pdf
        .groupby("seriousness")
        .agg(
            cnt=("case_id", "count"),
            avg_age=("age", "mean"),
            avg_dose=("dose", "mean"),
            avg_time_to_onset=("time_to_onset_days", "mean")
        )
        .sort_values("cnt", ascending=False)
    ),
    "Pandas: groupby seriousness"
)
pandas_result_1

Pandas: groupby seriousness
Среднее время: 0.1798 секунд
Все замеры: [0.1843, 0.1727, 0.1778, 0.1875, 0.1767]
--------------------------------------------------


,cnt,avg_age,avg_dose,avg_time_to_onset
seriousness,,,,
Non-serious,649378,46.357651,66.087401,100.510649
Hospitalization - initial or prolonged,58805,46.454655,65.045438,100.755140
Disability,58504,46.341430,65.305073,99.875205
Congenital anomaly,58501,46.297927,64.615801,100.644519
Death,58422,46.227774,66.038753,100.022355
Life-threatening,58202,46.166627,65.587059,101.150167
Other medically important condition,58188,46.303035,64.990771,101.409483


In [7]:
polars_result_1 = benchmark_many(
    lambda: (
        pldf
        .group_by("seriousness")
        .agg(
            pl.len().alias("cnt"),
            pl.col("age").mean().alias("avg_age"),
            pl.col("dose").mean().alias("avg_dose"),
            pl.col("time_to_onset_days").mean().alias("avg_time_to_onset")
        )
        .sort("cnt", descending=True)
    ),
    "Polars: group_by seriousness"
)
polars_result_1

Polars: group_by seriousness
Среднее время: 0.0409 секунд
Все замеры: [0.0443, 0.0408, 0.0498, 0.0341, 0.0354]
--------------------------------------------------


seriousness,cnt,avg_age,avg_dose,avg_time_to_onset
str,u32,f64,f64,f64
"""Non-serious""",649378,46.357651,66.087401,100.510649
"""Hospitalization - initial or p…",58805,46.454655,65.045438,100.75514
"""Disability""",58504,46.34143,65.305073,99.875205
"""Congenital anomaly""",58501,46.297927,64.615801,100.644519
"""Death""",58422,46.227774,66.038753,100.022355
"""Life-threatening""",58202,46.166627,65.587059,101.150167
"""Other medically important cond…",58188,46.303035,64.990771,101.409483


In [8]:
pandas_result_2 = benchmark_many(
    lambda: (
        pdf[pdf["seriousness"] == "Serious"]
        .groupby("suspect_drug")
        .agg(
            cnt=("case_id", "count"),
            avg_age=("age", "mean"),
            avg_bilirubin=("bilirubin_mg_dl", "mean"),
            avg_creatinine=("creatinine_mg_dl", "mean")
        )
        .sort_values("cnt", ascending=False)
        .head(20)
    ),
    "Pandas: filter Serious + groupby suspect_drug"
)
pandas_result_2

Pandas: filter Serious + groupby suspect_drug
Среднее время: 0.0779 секунд
Все замеры: [0.0786, 0.0795, 0.0777, 0.078, 0.0759]
--------------------------------------------------


,cnt,avg_age,avg_bilirubin,avg_creatinine
suspect_drug,,,,


In [9]:
polars_result_2 = benchmark_many(
    lambda: (
        pldf
        .filter(pl.col("seriousness") == "Serious")
        .group_by("suspect_drug")
        .agg(
            pl.len().alias("cnt"),
            pl.col("age").mean().alias("avg_age"),
            pl.col("bilirubin_mg_dl").mean().alias("avg_bilirubin"),
            pl.col("creatinine_mg_dl").mean().alias("avg_creatinine")
        )
        .sort("cnt", descending=True)
        .head(20)
    ),
    "Polars: filter Serious + group_by suspect_drug"
)
polars_result_2

Polars: filter Serious + group_by suspect_drug
Среднее время: 0.0037 секунд
Все замеры: [0.0068, 0.003, 0.0028, 0.0028, 0.0029]
--------------------------------------------------


suspect_drug,cnt,avg_age,avg_bilirubin,avg_creatinine
str,u32,f64,f64,f64


In [10]:
def benchmark_many(func, name, repeats=5):
    times = []

    for _ in range(repeats):
        start = time.perf_counter()
        result = func()
        end = time.perf_counter()
        times.append(end - start)

    avg_time = sum(times) / len(times)

    print(f"{name}")
    print(f"Среднее время: {avg_time:.4f} секунд")
    print(f"Все замеры: {[round(t, 4) for t in times]}")
    print("-" * 60)

    return result

In [11]:
pandas_complex_1 = benchmark_many(
    lambda: (
        pdf[
            (pdf["seriousness"] == "Serious") &
            (pdf["age"].notna()) &
            (pdf["dose"].notna()) &
            (pdf["creatinine_mg_dl"].notna())
        ]
        .groupby(["suspect_drug", "outcome"])
        .agg(
            cnt=("case_id", "count"),
            avg_age=("age", "mean"),
            median_age=("age", "median"),
            avg_dose=("dose", "mean"),
            max_dose=("dose", "max"),
            avg_creatinine=("creatinine_mg_dl", "mean"),
            avg_bilirubin=("bilirubin_mg_dl", "mean")
        )
        .sort_values("cnt", ascending=False)
        .head(20)
    ),
    "Pandas: complex filter + groupby drug/outcome"
)
pandas_complex_1

Pandas: complex filter + groupby drug/outcome
Среднее время: 0.0861 секунд
Все замеры: [0.087, 0.0856, 0.086, 0.0873, 0.0847]
------------------------------------------------------------


,,cnt,avg_age,median_age,avg_dose,max_dose,avg_creatinine,avg_bilirubin
suspect_drug,outcome,,,,,,,


In [12]:
polars_complex_1 = benchmark_many(
    lambda: (
        pldf
        .filter(
            (pl.col("seriousness") == "Serious") &
            pl.col("age").is_not_null() &
            pl.col("dose").is_not_null() &
            pl.col("creatinine_mg_dl").is_not_null()
        )
        .group_by(["suspect_drug", "outcome"])
        .agg(
            pl.len().alias("cnt"),
            pl.col("age").mean().alias("avg_age"),
            pl.col("age").median().alias("median_age"),
            pl.col("dose").mean().alias("avg_dose"),
            pl.col("dose").max().alias("max_dose"),
            pl.col("creatinine_mg_dl").mean().alias("avg_creatinine"),
            pl.col("bilirubin_mg_dl").mean().alias("avg_bilirubin")
        )
        .sort("cnt", descending=True)
        .head(20)
    ),
    "Polars: complex filter + group_by drug/outcome"
)
polars_complex_1

Polars: complex filter + group_by drug/outcome
Среднее время: 0.0058 секунд
Все замеры: [0.0068, 0.0066, 0.0064, 0.0052, 0.0039]
------------------------------------------------------------


suspect_drug,outcome,cnt,avg_age,median_age,avg_dose,max_dose,avg_creatinine,avg_bilirubin
str,str,u32,f64,f64,f64,i64,f64,f64


In [13]:
def pandas_complex_transform():
    temp = pdf.copy()

    temp["age_segment"] = pd.cut(
        temp["age"],
        bins=[0, 30, 45, 60, 75, 120],
        labels=["<30", "30-45", "45-60", "60-75", "75+"],
        right=False
    )

    temp["high_dose_flag"] = temp["dose"] > temp["dose"].median()
    temp["high_creatinine_flag"] = temp["creatinine_mg_dl"] > 1.2

    result = (
        temp
        .groupby(["age_segment", "seriousness", "high_creatinine_flag"], observed=True)
        .agg(
            cnt=("case_id", "count"),
            avg_dose=("dose", "mean"),
            avg_alt=("alt_u_l", "mean"),
            avg_ast=("ast_u_l", "mean"),
            avg_bun=("bun_mg_dl", "mean"),
            serious_rate=("high_dose_flag", "mean")
        )
        .sort_values("cnt", ascending=False)
        .head(30)
    )

    return result


pandas_complex_2 = benchmark_many(
    pandas_complex_transform,
    "Pandas: create features + groupby"
)
pandas_complex_2

Pandas: create features + groupby
Среднее время: 0.7491 секунд
Все замеры: [1.0066, 0.6609, 0.7158, 0.6894, 0.6728]
------------------------------------------------------------


cnt  \
age_segment seriousness                            high_creatinine_flag           
<30         Non-serious                            False                 174305   
45-60       Non-serious                            False                 108836   
30-45       Non-serious                            False                 108816   
75+         Non-serious                            False                  98943   
60-75       Non-serious                            False                  89967   
<30         Non-serious                            True                   20571   
            Life-threatening                       False                  15712   
            Congenital anomaly                     False                  15676   
            Hospitalization - initial or prolonged False                  15640   
            Death                                  False                  15630   
            Disability                             False                  15563   
            Other medically important condition    False                  15514   
45-60       Non-serious                            True                   12913   
30-45       Non-serious                            True                   12703   
75+         Non-serious                            True                   11735   
60-75       Non-serious                            True                   10589   
30-45       Hospitalization - initial or prolonged False                  10029   
            Death                                  False                   9922   
45-60       Congenital anomaly                     False                   9920   
30-45       Life-threatening                       False                   9887   
            Other medically important condition    False                   9876   
            Disability                             False                   9848   
45-60       Hospitalization - initial or prolonged False                   9836   
            Death                                  False                   9830   
30-45       Congenital anomaly                     False                   9818   
45-60       Disability                             False                   9808   
            Other medically important condition    False                   9673   
            Life-threatening                       False                   9648   
75+         Disability                             False                   8932   
            Hospitalization - initial or prolonged False                   8925   

                                                                          avg_dose  \
age_segment seriousness                            high_creatinine_flag              
<30         Non-serious                            False                 65.954287   
45-60       Non-serious                            False                 66.358264   
30-45       Non-serious                            False                 65.664222   
75+         Non-serious                            False                 65.452220   
60-75       Non-serious                            False                 66.156368   
<30         Non-serious                            True                  68.138447   
            Life-threatening                       False                 64.270303   
            Congenital anomaly                     False                 66.235009   
            Hospitalization - initial or prolonged False                 64.152941   
            Death                                  False                 64.874344   
            Disability                             False                 64.954636   
            Other medically important condition    False                 63.098105   
45-60       Non-serious                            True                  66.575312   
30-45       Non-serious                            True                  67.779816   
75+         Non-serious                    

In [14]:
def polars_complex_transform():
    dose_median = pldf.select(pl.col("dose").median()).item()

    result = (
        pldf
        .with_columns(
            pl.when(pl.col("age") < 30).then(pl.lit("<30"))
              .when(pl.col("age") < 45).then(pl.lit("30-45"))
              .when(pl.col("age") < 60).then(pl.lit("45-60"))
              .when(pl.col("age") < 75).then(pl.lit("60-75"))
              .otherwise(pl.lit("75+"))
              .alias("age_segment"),

            (pl.col("dose") > dose_median).alias("high_dose_flag"),
            (pl.col("creatinine_mg_dl") > 1.2).alias("high_creatinine_flag")
        )
        .group_by(["age_segment", "seriousness", "high_creatinine_flag"])
        .agg(
            pl.len().alias("cnt"),
            pl.col("dose").mean().alias("avg_dose"),
            pl.col("alt_u_l").mean().alias("avg_alt"),
            pl.col("ast_u_l").mean().alias("avg_ast"),
            pl.col("bun_mg_dl").mean().alias("avg_bun"),
            pl.col("high_dose_flag").mean().alias("high_dose_rate")
        )
        .sort("cnt", descending=True)
        .head(30)
    )

    return result


polars_complex_2 = benchmark_many(
    polars_complex_transform,
    "Polars: create features + group_by"
)
polars_complex_2

Polars: create features + group_by
Среднее время: 0.1874 секунд
Все замеры: [0.2506, 0.1985, 0.1767, 0.1611, 0.1502]
------------------------------------------------------------


age_segment,seriousness,high_creatinine_flag,cnt,avg_dose,avg_alt,avg_ast,avg_bun,high_dose_rate
str,str,bool,u32,f64,f64,f64,f64,f64
"""<30""","""Non-serious""",false,125825,66.086819,61.300504,56.852038,13.487138,0.336563
"""30-45""","""Non-serious""",false,78389,66.134534,61.060058,56.316734,13.510445,0.339078
"""45-60""","""Non-serious""",false,78325,66.669071,61.790328,56.457537,13.507669,0.335742
"""75+""","""Non-serious""",false,71393,65.248792,61.647057,57.329681,13.502246,0.337414
"""60-75""","""Non-serious""",false,64647,66.795629,60.669207,56.670506,13.48835,0.338948
…,…,…,…,…,…,…,…,…
"""45-60""","""Hospitalization - initial or p…",false,7095,62.209443,64.254858,61.18264,13.556969,0.330374
"""30-45""","""Other medically important cond…",false,7077,66.252791,61.061037,56.6031,13.486942,0.336725
"""45-60""","""Disability""",false,7062,63.135231,63.289381,57.920865,13.6118,0.335599


In [15]:
pandas_complex_3 = benchmark_many(
    lambda: (
        pdf
        .groupby(["country", "sex", "age_group", "suspect_drug", "seriousness"])
        .agg(
            cnt=("case_id", "count"),
            avg_age=("age", "mean"),
            avg_weight=("weight_kg", "mean"),
            avg_dose=("dose", "mean"),
            median_duration=("event_duration_days", "median"),
            avg_alt=("alt_u_l", "mean"),
            avg_ast=("ast_u_l", "mean"),
            avg_bilirubin=("bilirubin_mg_dl", "mean"),
            avg_creatinine=("creatinine_mg_dl", "mean"),
            avg_bun=("bun_mg_dl", "mean")
        )
        .sort_values("cnt", ascending=False)
        .head(50)
    ),
    "Pandas: many keys + many metrics"
)
pandas_complex_3

Pandas: many keys + many metrics
Среднее время: 1.1138 секунд
Все замеры: [0.7244, 0.9157, 1.1937, 1.3795, 1.3555]
------------------------------------------------------------


,,,,,cnt,avg_age,avg_weight,avg_dose,median_duration,avg_alt,avg_ast,avg_bilirubin,avg_creatinine,avg_bun
country,sex,age_group,suspect_drug,seriousness,,,,,,,,,,
ZAF,Male,adult,Psychostabil,Non-serious,216,42.097222,86.265772,79.898148,43.5,46.521472,41.090361,1.100000,1.103247,14.277778
FIN,Male,adult,Pulmofix,Non-serious,215,40.883721,85.719149,65.572093,37.5,50.580838,60.475610,1.255488,1.248428,18.144737
CAN,Male,adult,Pulmofix,Non-serious,215,43.637209,84.850350,44.069767,44.0,62.581699,54.634146,1.261310,1.240361,17.915663
USA,Female,adult,Dermatopin,Non-serious,214,41.747664,84.222794,59.542056,43.0,67.161491,50.796407,1.183436,1.157792,17.218935
TUR,Male,adult,Sleepwell,Non-serious,212,41.613208,84.446259,56.896226,40.0,52.127389,64.438272,1.283544,1.182927,17.617143
GRC,Female,adult,Oncotreat,Non-serious,212,39.127358,83.742336,52.976415,35.5,66.940476,59.030488,1.126582,1.140854,16.768293
NZL,Unknown,adult,Osteostrong,Non-serious,210,40.890476,84.419708,68.095238,41.0,54.622517,50.854430,1.158710,1.352667,17.186335
CHN,Unknown,adult,Gastroease,Non-serious,209,41.559809,84.156250,72.937799,39.0,66.769737,51.808917,1.142308,1.129677,16.603896
BEL,Male,adult,Nephroguard,Non-serious,209,43.717703,86.992908,41.287081,50.0,54.069620,48.616883,1.155844,1.094872,16.278481


In [16]:
polars_complex_3 = benchmark_many(
    lambda: (
        pldf
        .group_by(["country", "sex", "age_group", "suspect_drug", "seriousness"])
        .agg(
            pl.len().alias("cnt"),
            pl.col("age").mean().alias("avg_age"),
            pl.col("weight_kg").mean().alias("avg_weight"),
            pl.col("dose").mean().alias("avg_dose"),
            pl.col("event_duration_days").median().alias("median_duration"),
            pl.col("alt_u_l").mean().alias("avg_alt"),
            pl.col("ast_u_l").mean().alias("avg_ast"),
            pl.col("bilirubin_mg_dl").mean().alias("avg_bilirubin"),
            pl.col("creatinine_mg_dl").mean().alias("avg_creatinine"),
            pl.col("bun_mg_dl").mean().alias("avg_bun")
        )
        .sort("cnt", descending=True)
        .head(50)
    ),
    "Polars: many keys + many metrics"
)
polars_complex_3

Polars: many keys + many metrics
Среднее время: 0.8684 секунд
Все замеры: [1.0782, 1.1393, 0.9312, 0.5949, 0.5984]
------------------------------------------------------------


country,sex,age_group,suspect_drug,seriousness,cnt,avg_age,avg_weight,avg_dose,median_duration,avg_alt,avg_ast,avg_bilirubin,avg_creatinine,avg_bun
str,str,str,str,str,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ZAF""","""Male""","""adult""","""Psychostabil""","""Non-serious""",216,42.097222,86.265772,79.898148,43.5,46.521472,41.090361,1.1,1.103247,14.277778
"""FIN""","""Male""","""adult""","""Pulmofix""","""Non-serious""",215,40.883721,85.719149,65.572093,37.5,50.580838,60.47561,1.255488,1.248428,18.144737
"""CAN""","""Male""","""adult""","""Pulmofix""","""Non-serious""",215,43.637209,84.85035,44.069767,44.0,62.581699,54.634146,1.26131,1.240361,17.915663
"""USA""","""Female""","""adult""","""Dermatopin""","""Non-serious""",214,41.747664,84.222794,59.542056,43.0,67.161491,50.796407,1.183436,1.157792,17.218935
"""TUR""","""Male""","""adult""","""Sleepwell""","""Non-serious""",212,41.613208,84.446259,56.896226,40.0,52.127389,64.438272,1.283544,1.182927,17.617143
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""AUT""","""Unknown""","""adult""","""Eyeprotect""","""Non-serious""",201,40.044776,87.630435,53.343284,41.0,52.298137,45.714286,1.031293,1.20068,15.870748
"""IND""","""Male""","""adult""","""Dermatopin""","""Non-serious""",201,40.661692,86.4928,61.960199,39.5,46.474026,43.569536,1.073856,1.281884,17.197452
"""POL""","""Female""","""adult""","""Urocare""","""Non-serious""",201,42.517413,86.481452,60.358209,41.0,63.149351,53.109589,1.279195,1.2,15.683453


In [17]:
benchmark_results = []

def benchmark_many_save(func, name, library, operation, repeats=5):
    times = []

    for _ in range(repeats):
        start = time.perf_counter()
        result = func()
        end = time.perf_counter()
        times.append(end - start)

    avg_time = sum(times) / len(times)

    benchmark_results.append({
        "library": library,
        "operation": operation,
        "avg_time_sec": round(avg_time, 4),
        "min_time_sec": round(min(times), 4),
        "max_time_sec": round(max(times), 4),
        "all_runs": [round(t, 4) for t in times]
    })

    print(f"{name}: {avg_time:.4f} секунд")
    return result

In [18]:
pandas_complex_1 = benchmark_many_save(
    lambda: (
        pdf[
            (pdf["seriousness"] == "Serious") &
            (pdf["age"].notna()) &
            (pdf["dose"].notna()) &
            (pdf["creatinine_mg_dl"].notna())
        ]
        .groupby(["suspect_drug", "outcome"])
        .agg(
            cnt=("case_id", "count"),
            avg_age=("age", "mean"),
            median_age=("age", "median"),
            avg_dose=("dose", "mean"),
            max_dose=("dose", "max"),
            avg_creatinine=("creatinine_mg_dl", "mean"),
            avg_bilirubin=("bilirubin_mg_dl", "mean")
        )
        .sort_values("cnt", ascending=False)
        .head(20)
    ),
    "Pandas complex 1",
    "Pandas",
    "complex filter + groupby"
)

polars_complex_1 = benchmark_many_save(
    lambda: (
        pldf
        .filter(
            (pl.col("seriousness") == "Serious") &
            pl.col("age").is_not_null() &
            pl.col("dose").is_not_null() &
            pl.col("creatinine_mg_dl").is_not_null()
        )
        .group_by(["suspect_drug", "outcome"])
        .agg(
            pl.len().alias("cnt"),
            pl.col("age").mean().alias("avg_age"),
            pl.col("age").median().alias("median_age"),
            pl.col("dose").mean().alias("avg_dose"),
            pl.col("dose").max().alias("max_dose"),
            pl.col("creatinine_mg_dl").mean().alias("avg_creatinine"),
            pl.col("bilirubin_mg_dl").mean().alias("avg_bilirubin")
        )
        .sort("cnt", descending=True)
        .head(20)
    ),
    "Polars complex 1",
    "Polars",
    "complex filter + groupby"
)

Pandas complex 1: 0.1023 секунд
Polars complex 1: 0.0043 секунд


In [19]:
benchmark_df = pd.DataFrame(benchmark_results)
benchmark_df

,library,operation,avg_time_sec,min_time_sec,max_time_sec,all_runs
0,Pandas,complex filter + groupby,0.1023,0.0981,0.1063,"[0.1063, 0.1027, 0.1014, 0.1032, 0.0981]"
1,Polars,complex filter + groupby,0.0043,0.0037,0.0052,"[0.0052, 0.0037, 0.0047, 0.0041, 0.0039]"


In [20]:
pandas_time = benchmark_df.loc[
    (benchmark_df["library"] == "Pandas") &
    (benchmark_df["operation"] == "complex filter + groupby"),
    "avg_time_sec"
].iloc[0]

polars_time = benchmark_df.loc[
    (benchmark_df["library"] == "Polars") &
    (benchmark_df["operation"] == "complex filter + groupby"),
    "avg_time_sec"
].iloc[0]

pandas_time / polars_time

np.float64(23.790697674418606)